> 在线讲义：[第 1 章 · 一个 kernel 怎么分给成千上万个线程：grid / block / thread 与全局编号](https://wikieden.github.io/cuda-to-tensor-core/lessons/01-thread-hierarchy.html) · 本书《CUDA 从线程到 Tensor Core》

# 第 1 章 · 一个 kernel 怎么分给成千上万个线程：grid / block / thread 与全局编号

本章回答：CPU 上一个 `for` 循环做完的事，放到 GPU 上怎么写？启动 kernel 时那串 `<<<4, 256>>>` 是什么意思？每个线程怎么知道自己该处理第几个数？

配套讲义：仓库里 `cuda-programming/notes/lessons/01-thread-hierarchy.md`（讲解、动画、练习都在讲义里）。对应原文：Programming Guide（PG）13.4.2 §1.2.2.1、§2.1.2.2、§2.1.2.3（`materials/text/cuda-programming-guide.txt` 第 1000–1058、1622–1760 行）；Oxford lec1 第 19–24 页（`materials/text/oxford-lec1.txt` 第 278–372 行）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二、三步 `<<<4, 256>>>` | $i = b \times 256 + t$ | `launch_indices(4, 256)`：广播出 4 × 256 的编号表 | block 0–3 各管 0–255、256–511、512–767、768–1023 |
| 第三步 Oxford 例子 | $1 \times 64 + 44$ | `global_index(1, 64, 44)` | 108 |
| 第四步 向上取整 | $\lceil 1000 / 256 \rceil$ | `ceil_div(1000, 256)`，即 `(n + B - 1) // B` | 4；开 1024 个线程，24 个越界 |
| 第四步 越界检查 | `if (i < n)` | `vec_add(a, b, 256)`：先 `i = i[i < n]` 再加 | 和 `a + b` 一样；去掉检查 → `IndexError` |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `ceil_div`**（来自 `ch01_thread_index.py`）：向上取整的整数除法：(n + b - 1) // b，即 C++ 里的 (n + B - 1) / B。

In [ ]:
def ceil_div(n: int, b: int) -> int:
    """向上取整的整数除法：(n + b - 1) // b，即 C++ 里的 (n + B - 1) / B。"""
    return (n + b - 1) // b

**函数 `global_index`**（来自 `ch01_thread_index.py`）：全局编号：i = blockIdx.x × blockDim.x + threadIdx.x。

In [ ]:
def global_index(block_idx: int, block_dim: int, thread_idx: int) -> int:
    """全局编号：i = blockIdx.x × blockDim.x + threadIdx.x。"""
    return block_idx * block_dim + thread_idx

**函数 `launch_indices`**（来自 `ch01_thread_index.py`）：模拟 <<<grid_dim, block_dim>>>：返回形状 (grid_dim, block_dim) 的表，第 b 行第 t 列是那个线程的 i。

In [ ]:
def launch_indices(grid_dim: int, block_dim: int) -> np.ndarray:
    """模拟 <<<grid_dim, block_dim>>>：返回形状 (grid_dim, block_dim) 的表，第 b 行第 t 列是那个线程的 i。"""
    block_idx = np.arange(grid_dim)[:, None]   # 每一行是一个 block
    thread_idx = np.arange(block_dim)[None, :]  # 每一列是 block 里的一个座位
    return block_idx * block_dim + thread_idx

**函数 `vec_add`**（来自 `ch01_thread_index.py`）：模拟 vecAdd kernel：host 端向上取整开 block，kernel 端每个线程做 if (i < n) c[i] = a[i] + b[i]。

In [ ]:
def vec_add(a: np.ndarray, b: np.ndarray, block_dim: int, bounds_check: bool = True) -> np.ndarray:
    """模拟 vecAdd kernel：host 端向上取整开 block，kernel 端每个线程做 if (i < n) c[i] = a[i] + b[i]。"""
    n = len(a)
    c = np.zeros_like(a)
    i = launch_indices(ceil_div(n, block_dim), block_dim).ravel()  # 所有线程的 i 排成一串
    if bounds_check:
        i = i[i < n]                                                # 越界的线程"举手不答题"
    c[i] = a[i] + b[i]                                              # 不检查时这里会读写数组外面
    return c

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二、三步 <<<4, 256>>>：每个 block 负责哪一段")
idx = launch_indices(4, 256)
for blk in range(4):
    print(f"  block {blk}: i = {idx[blk, 0]} … {idx[blk, -1]}")
print("  总线程数 =", idx.size, "；不重不漏：", np.array_equal(np.sort(idx.ravel()), np.arange(1024)))

In [ ]:
print("\n第三步 Oxford 例子：第 1 号 block（每个 64 线程）里的第 44 号线程")
print("  i =", global_index(1, 64, 44))

In [ ]:
print("\n第四步 n = 1000、block = 256：向上取整 + 越界检查")
n, B = 1000, 256
print(f"  n / B = {n / B:.2f}，整数除法 n // B = {n // B}，向上取整 = {ceil_div(n, B)}")
print(f"  开了 {ceil_div(n, B) * B} 个线程，其中 {ceil_div(n, B) * B - n} 个越界，什么也不做")
rng = np.random.default_rng(0)
a, b = rng.random(n, dtype=np.float32), rng.random(n, dtype=np.float32)
print("  加了越界检查，结果和 a + b 一样：", np.allclose(vec_add(a, b, B), a + b))
try:
    vec_add(a, b, B, bounds_check=False)
except IndexError as e:
    print("  去掉越界检查：NumPy 直接报错（GPU 上不会报错，而是悄悄读写别处的内存）→", type(e).__name__)

## 3. 代码里才看得到的

1. **"一次启动所有线程"在代码里就是一次广播**：`launch_indices` 返回一张 grid × block 的表，第 b 行第 t 列就是 `blockIdx.x = b`、`threadIdx.x = t` 的那个线程算出的 i。GPU 上这些线程真的同时跑，NumPy 只是一次算完整张表。
2. **越界在 NumPy 里会报错，在 GPU 上不会**：去掉越界检查时 NumPy 抛 `IndexError`；GPU 上没有这层保护，越界读写会悄悄碰到别的内存（第 4 章用 compute-sanitizer 查）。
3. **Python 的 `//` 是向下取整**：`1000 // 256` 是 3，所以向上取整要写成 `(n + B - 1) // B`，和 C++ 的整数除法同一个套路。

## 4. 自己改一改

- 把 `launch_indices(4, 256)` 换成 `launch_indices(4, 8)`，打印出来看看每个 block 管哪几个数，和动画里 block 大小 8 的画面对一对。
- 把第四步的 n 改成 1024、2000，看越界线程数怎么变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 只写循环体，一人领一个；
> 学校分教室，教室分座位：grid 套 block，block 套 thread；
> 准考证号 = 教室号 × 每间座位数 + 座位号；
> 人数向上取整，多的人举手不答题（`if (i < n)`）。